## TEST MODELS INITIALISATION

In [ ]:
import sys
sys.path.append('../')

import time
import copy
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

from dataclasses import dataclass

import matplotlib.pyplot as plt

import models
import importlib
importlib.reload(models)

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)

## LOAD A SIMPLE TEXT DATASET TO TEST LANGUAGE MODELS

In [ ]:
num_sequences = 8
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("gpt2")
# vocab_size = tokenizer.vocab_size
# print(vocab_size)

import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.max_token_value+1
print("vocabulary size:", vocab_size)

tokens = tokenizer.encode("Hello, I'm a language model, ")
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(num_sequences, 1).to(device) # size B, T
print(tokens.size())

In [ ]:
model = models.CLM(
    vocab_size=vocab_size,
    block_size=128,
    embedding_dim=768,
    num_heads=12,
    ffwd_size=4,
    num_layers=12,
    dropout=0,
    share_emb=False
)
model.to(device)
param_count = sum([p.numel() for p in model.parameters()])
print("# parameters:", param_count)

model.eval()
with torch.no_grad():
    generated = model.generate(tokens, num_tokens=32)
model.train()

In [ ]:
for i in range(num_sequences):
    print(tokenizer.decode(generated[i].tolist()))

In [ ]:
print(model.token_embedding_table.weight.size())
print(model.lm_head.weight.size())
# print(param_count - 2*50257*768)

In [ ]:
# Test model with RoPE
model = models.CLM(
    vocab_size=vocab_size,
    block_size=128,
    embedding_dim=768,
    num_heads=12,
    ffwd_size=4,
    num_layers=12,
    dropout=0.,
    rope=True,
    share_emb=False
)
model.to(device)
param_count = sum([p.numel() for p in model.parameters()])
print("# parameters:", param_count)

model.eval()
with torch.no_grad():
    generated = model.generate(tokens, num_tokens=32)
model.train()

In [ ]:
for i in range(num_sequences):
    print(tokenizer.decode(generated[i].tolist()))

## TEST ROTARY POSITIONAL ENCODING

In [ ]:
class RotaryEncoding(nn.Module):
    def __init__(
        self, head_dim: int, base: float = 10000.0, offset: int = 0
    ):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head dimension"

        self.dim = head_dim
        self.base = base
        self.offset = offset

        # theta (head_dim/2)
        theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("theta", theta, persistent=False)

    def _rotate_half(
        self, x: torch.Tensor
    ) -> torch.Tensor:
        # x: (..., head_sim) where head_dim is even
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).reshape_as(x)

    def _get_cos_sin(
        self, seq_len: int, device: torch.device, dtype: torch.dtype
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # positions (T)
        pos = torch.arange(self.offset, self.offset + seq_len, device=device, dtype=dtype)
        pos_theta = pos[:,None] @ self.theta[None]
        # pos_theta -> (T, head_dim) by duplicating each freq for the pair
        pos_theta = torch.repeat_interleave(pos_theta, repeats=2, dim=-1)
        pos_cos = pos_theta.cos().to(dtype=dtype)[None, None, :, :]
        pos_sin = pos_theta.sin().to(dtype=dtype)[None, None, :, :]
        return pos_cos, pos_sin

    def forward(
        self, q: torch.Tensor, k: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            q: The query, tensor of size (batch_size, num_heads, seq_len, head_dim).
            k: The key, tensor of size (batch_size, num_heads, seq_len, head_dim).

        Returns:
            rotated q and k matrices,
            of size (batch_size, num_heads, seq_len, head_dim)
        """
        B,H,T,C = q.size()
        # compute sines and cosines
        cos, sin = self._get_cos_sin(T,device=q.device,dtype=q.dtype)
        q_r = (q * cos) + (self._rotate_half(q) * sin)
        k_r = (k * cos) + (self._rotate_half(k) * sin)

        return q_r, k_r

In [ ]:
seq_len = 256
vocab_size = 8192
bs = 32

head_dim = 64
num_heads = 12

x = torch.randint( vocab_size, (bs, seq_len), device=device)
print('input size:', x.size())
random_embedding = nn.Embedding(vocab_size, head_dim*num_heads)
random_embedding = random_embedding.to(device)
x_emb = random_embedding(x)
print('embedding size:', x_emb.size())

key = torch.randn( head_dim*num_heads, head_dim*num_heads).to(device)
k = F.linear( x_emb, key, bias=None).view(bs, seq_len, num_heads, head_dim).transpose(1,2) * (head_dim*num_heads)**-.5    # [bs, num_heads, seq_len, head_dim]
query = torch.randn( head_dim*num_heads, head_dim*num_heads).to(device)
q = F.linear( x_emb, query, bias=None).view(bs, seq_len, num_heads, head_dim).transpose(1,2) * (head_dim*num_heads)**-.5  # [bs, num_heads, seq_len, head_dim]
print('Q/K size:', q.size(), k.size(), '(B,H,T,C)')

In [ ]:
RoPE = RotaryEncoding( head_dim).to(device)
print(RoPE)

q_r, k_r = RoPE(q,k)
print('rotated Q/K size:', q_r.size(), k_r.size())
# q and q_r are only equal in position 0 because there is no rotation there
print((q_r[:,:,0,:]==q[:,:,0,:]).sum()/(bs*num_heads*head_dim))
print((q_r[:,:,3,:]==q[:,:,3,:]).sum())

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multiple Attention Heads.
    
    Args:
        input_dim: The dimension of input tokens.
        input_size: The (maximal) number of input tokens.
        num_heads: The number of heads.
        out_dim: The dimension of output tokens.
        dropout: The fraction of weights to zero via dropout.
        decoder: True for one-directional attention.
        
    """
    def __init__(
        self, input_dim, input_size, num_heads, out_dim, dropout, decoder=True, rope=False
    ):
        super().__init__()
        assert out_dim % num_heads == 0, "inner dim. must be multiple of num. heads"

        self.input_dim = input_dim
        self.input_size = input_size
        self.out_dim = out_dim
        self.num_heads = num_heads
        self.head_dim = out_dim // num_heads
        self.dropout = dropout
        self.decoder = decoder
        self.rope = rope

        self.key=nn.Parameter(
            torch.randn( self.out_dim, self.input_dim)
        )
        self.query=nn.Parameter(
            torch.randn( self.out_dim, self.input_dim)
        )
        self.value=nn.Parameter(
            torch.randn( self.out_dim, self.input_dim)
        )
        self.projection=nn.Parameter(
            torch.randn( self.out_dim, self.out_dim)
        )
        self.dropout = nn.Dropout(self.dropout)
        if rope:
            self.rotary_emb = RotaryEncoding(self.head_dim)
        if decoder:
            self.register_buffer('tril', torch.tril(torch.ones(self.input_size, self.input_size)))
        else:
            self.register_buffer('tril', torch.ones(self.input_size, self.input_size))

    def forward(self, x):
        """
        Args:
            x: input, tensor of size (batch_size, input_size, input_dim).
        
        Returns:
            The output of a multi-head attention layer,
            of size (batch_size, input_size, output_dim)
        """
        B,T,C = x.size()
        k = F.linear( x, self.key, bias=None).view(B, T, self.num_heads, self.head_dim).transpose(1,2) * C**-.5    # [bs, num_heads, seq_len, head_dim]
        q = F.linear( x, self.query, bias=None).view(B, T, self.num_heads, self.head_dim).transpose(1,2) * C**-.5  # [bs, num_heads, seq_len, head_dim]
        v = F.linear( x, self.value, bias=None).view(B, T, self.num_heads, self.head_dim).transpose(1,2) * C**-.5  # [bs, num_heads, seq_len, head_dim]
        if self.rope:
            q, k = self.rotary_emb(q, k)

        weight = q @ k.transpose(-2,-1) * self.head_dim**-.5             # [bs, num_heads, seq_len, seq_len]
        weight = weight.masked_fill(self.tril[:T,:T]==0, float('-inf'))  #  //
        weight = F.softmax(weight, dim=-1)                               #  //
        weight = self.dropout(weight)

        out = (weight @ v).transpose(1,2).reshape(B,T,-1) # [bs, seq_len, out_dim]
        # out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        # out = out.transpose(1,2).contiguous().view(B,T,C)
        out = F.linear( out, self.projection, bias=None) * self.projection.size(-1)**-.5

        return out

In [ ]:
seq_len = 256
vocab_size = 8192
bs = 32

head_dim = 64
num_heads = 12

x = torch.randint( vocab_size, (bs, seq_len), device=device)
print('input size:', x.size())
random_embedding = nn.Embedding(vocab_size, head_dim*num_heads)
random_embedding = random_embedding.to(device)
x_emb = random_embedding(x)
print('embedding size:', x_emb.size())

In [ ]:
rope_attn = MultiHeadAttention(
    input_dim=num_heads*head_dim,
    input_size=seq_len, 
    num_heads=num_heads, 
    out_dim=num_heads*head_dim, 
    dropout=0, 
    decoder=True,
    rope=True
).to(device)
print(rope_attn)

with torch.no_grad():
    x_attn = rope_attn(x_emb)
print('output size:', x_attn.size())